In [2]:
import os
import sys
import json
import torch
import pickle
import warnings
import numpy as np
import sympy as sp
import pandas as pd
import proplot as pplt
from IPython.display import display,HTML
sys.path.insert(0,'..')
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [3]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
MODELSDIR = CONFIGS['filepaths']['models']
SRCONFIG  = CONFIGS['experiments']['sr']
SEEDS     = SRCONFIG['seeds']

In [4]:
VARDICT = {
    'rh':r'\widehat{\mathrm{RH}}',
    'thetae':r'\widehat{\theta}_{e}',
    'thetaestar':r'{\widehat{\theta}_{e}^{*}}',
    'bl':r'\widetilde{\mathrm{B_L}}',
    'lf':r'\widetilde{\mathrm{LF}}',
    'shf':r'\widetilde{\mathrm{SHF}}',
    'lhf':r'\widetilde{\mathrm{LHF}}'}
SYMBOLS  = {k:sp.Symbol(k) for k in VARDICT}
FUNCDICT = {
    'min':sp.Min,
    'max':sp.Max,
    'abs':sp.Abs,
    'neg':lambda x:-x,
    'square':lambda x:x**2,
    'cube':lambda x:x**3,
    'exp':sp.exp}

TERMORDER = {'bl':0,'rh':1,'thetae':2,'thetaestar':3,'lf':4,'shf':5,'lhf':6}

def to_sympy(eq):
    return sp.sympify(eq,locals={**SYMBOLS,**FUNCDICT})

def round_nums(expr,ndigits=4):
    return expr.xreplace({n:sp.Float(round(float(n),ndigits),ndigits) for n in expr.atoms(sp.Float)})

def term_key(term):
    syms = term.free_symbols
    if not syms:return (99,str(term))
    names = sorted(s.name for s in syms)
    return (min(TERMORDER.get(n,50) for n in names),str(term))

def order_expr(expr):
    if expr.args:
        expr = expr.func(*[order_expr(arg) for arg in expr.args],evaluate=False)
    if isinstance(expr,sp.Add):
        expr = sp.Add(*sorted(sp.Add.make_args(expr),key=term_key),evaluate=False)
    return expr

def to_latex(expr):
    symnames = {SYMBOLS[k]:v for k,v in VARDICT.items()}
    latex = sp.latex(expr,symbol_names=symnames,mul_symbol='dot')
    latex = latex.replace(r'\left','').replace(r'\right','')
    return ' '.join(latex.split())

def prettify(eq):
    try:
        expr = to_sympy(str(eq).strip())
        expr = round_nums(expr)
        expr = order_expr(expr)
        return '$'+to_latex(expr)+'$'
    except Exception:
        return str(eq).strip()

def load_equations(runname):
    seedframes = {}
    for seed in SEEDS:
        filepath = os.path.join(MODELSDIR,'sr',f'{runname}_{seed}_equations.csv')
        if not os.path.exists(filepath):continue
        df = pd.read_csv(filepath)
        df['seed'] = seed
        seedframes[seed] = df
    return seedframes

def load_registry():
    filepath = os.path.join(MODELSDIR,'sr','optimized_equations.pkl')
    if not os.path.exists(filepath):return {}
    with open(filepath,'rb') as f:return pickle.load(f)

def prettify_optimized(opt):
    ns = {**SYMBOLS,**FUNCDICT}
    ns.update({k:sp.Float(v) for k,v in opt['constants'].items()})
    try:
        expr = eval(opt['form'],{'__builtins__':{}},ns)
        expr = round_nums(expr)
        expr = order_expr(expr)
        return r'$'+to_latex(expr)+'$'
    except Exception as e:
        return f"{opt['form']}  [render error: {e}]"

In [5]:
registry = load_registry()

CONSTANTMAP = {
    'sr_lo':{'a':'alpha','b':'beta'},
    'sr_bl':{'a':'beta1','b':'beta2'},
    'sr_med':{'a':'alpha','b':'gamma','c':'eta'},
    'sr_hi':{'a':'alpha','b':'gamma','c':'eta','d':'lambda'}}
GREEKCOLS   = ['alpha','beta','beta1','beta2','gamma','eta','lambda']
COLLABELS   = {
    'alpha':r'$\alpha$',  'beta':  r'$\beta$',
    'beta1':r'$\beta_1$', 'beta2': r'$\beta_2$',
    'gamma':r'$\gamma$',  'eta':   r'$\eta$',
    'lambda':r'$\delta$',}
GREEK_SYMS   = {
    'alpha': sp.Symbol('alpha'), 'beta':  sp.Symbol('beta'),
    'beta1': sp.Symbol('beta_1'), 'beta2': sp.Symbol('beta_2'),
    'gamma': sp.Symbol('gamma'),  'eta':   sp.Symbol('eta'),
    'delta': sp.Symbol('delta'),
}
MODEL_ORDER  = ['sr_lo', 'sr_bl', 'sr_med', 'sr_hi']

# ── Table 1: optimized constants ──────────────────────────────────────────────
rows1 = []
for name in MODEL_ORDER:
    eqspec = SRCONFIG['optimizedeqs'].get(name)
    if eqspec is None: continue
    opt  = registry.get(name)
    cmap = CONST_MAP.get(name, {})
    row  = {'Equation': eqspec['description']}
    for col in GREEK_COLS:
        internal = next((k for k, v in cmap.items() if v == col), None)
        row[COL_LABELS[col]] = (f"{opt['constants'][internal]:.4f}"
                                if internal and opt and internal in opt['constants']
                                else '\u2013')
    rows1.append(row)
display(HTML(pd.DataFrame(rows1).set_index('Equation').to_html(escape=False)))

# ── Table 2: symbolic equation forms ─────────────────────────────────────────
def prettify_symbolic(name, opt):
    cmap = CONST_MAP.get(name, {})
    ns   = {**SYMBOLS, **FUNCDICT}
    for internal, greek in cmap.items():
        ns[internal] = GREEK_SYMS[greek]
    try:
        expr = eval(opt['form'], {'__builtins__': {}}, ns)
        expr = order_expr(expr)
        return '$' + to_latex(expr) + '$'
    except Exception as e:
        return f"{opt['form']}  [error: {e}]"

rows2 = []
for name in MODEL_ORDER:
    eqspec = SRCONFIG['optimizedeqs'].get(name)
    if eqspec is None: continue
    opt  = registry.get(name)
    rows2.append({'Equation': eqspec['description'],
                  'Form': prettify_symbolic(name, opt) if opt else '\u2013'})
display(HTML(pd.DataFrame(rows2).set_index('Equation').to_html(escape=False)))

,$\alpha$,$\beta$,$\beta_1$,$\beta_2$,$\gamma$,$\eta$,$\delta$
Equation,,,,,,,
SR-LO,1.0000,-0.6507,–,–,–,–,–
SR-BL,–,–,0.2957,0.1421,–,–,–
SR-MED,1.5576,–,–,–,1.4706,0.3756,–
SR-HI,1.3070,–,–,–,-1.3594,-0.4173,-0.1271


,Form
Equation,
SR-LO,$\alpha \cdot e^{\widehat{\mathrm{RH}}} + \beta$
SR-BL,$\beta_{2} + (\beta_{1} + \widetilde{\mathrm{B_L}})^{3}$
SR-MED,"$\alpha \cdot \max(\widehat{\mathrm{RH}}, - \eta - \gamma \cdot {\widehat{\theta}_{e}^{*}} + \widehat{\theta}_{e})^{3}$"
SR-HI,"$(\alpha \cdot \max(\widehat{\mathrm{RH}}, \eta + \gamma \cdot {\widehat{\theta}_{e}^{*}} + \widehat{\theta}_{e}) + \delta \cdot \max(\widetilde{\mathrm{LF}}, \widetilde{\mathrm{SHF}}))^{3}$"
